# 13 - Robustez del ajuste de hiperparámetros
En el notebook 12 hemos visto que en los modelos de árboles (`random_forest`, `lightgbm` y `xgboost_balanced`), el ajuste de hiperparámetrosno ha mejorado a los valores por defecto. Vamos a intentar comprobar por qué. Puede ser por dos motivos:
1. La búsqueda se queda corta, porque solo habíamos probado 30 candidatas al azar.
2. La validación interna es muy ruidosa, porque la clase A tiene unos 24 tramos por ciudad. La candidata que gana puede ser la que ha tenido más suerte, y entonces una búsqueda mejor no ayudaría.

Vamos a repetir el ajuste de hiperparámetros de los árboles de dos maneras y lo comparamos con el ajuste principal (al que llamamos A).
- **B**, con otra seed, que cambia los pliegues internos y las candidatas. Si el resultado cambia mucho, apunta a la segunda explicación.
- **C**, con 100 candidatas, en lugar de 30. Si el test no mejora, la primera explicación no se sostiene.

No vamos a repetir la regresión logística, porque su rejilla siempre prueba las mismas combinaciones, y los resultados han sido mejores.

Los ajustes se lanzan con `make tune-variants` (`urban_accessibility_ml/modeling/tuning/tuning_variants.py`), así que aquí solo se leen los resultados.

## Carga de los resultados

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from urban_accessibility_ml.config import REPORTS_DIR, SEED
from urban_accessibility_ml.modeling.tuning.tuning_constants import TUNING_N_CANDIDATES
from urban_accessibility_ml.modeling.tuning.tuning_variants import TREE_MODELS, VARIANTS, VARIANTS_DIR
from urban_accessibility_ml.plotting.palettes import SERIE_BASE
from urban_accessibility_ml.plotting.plots import SERIES, save_figure, use_style

RESULTS = REPORTS_DIR / "results"
KEY = ["strategy", "name", "fold", "feature_set", "profile", "model"]
# One colour per model, the same as in notebook 11
MODEL_COLORS = {"random_forest": SERIES[0], "xgboost_balanced": SERIES[1], "lightgbm": SERIES[2]}

use_style()

default = pd.read_csv(RESULTS / "experiment-matrix-default.csv")
matrices = {
    "A (30 candidatas)": pd.read_csv(RESULTS / "experiment-matrix-tuned.csv"),
    "B (otra seed)": pd.read_csv(VARIANTS_DIR / "experiment-matrix_seed2.csv"),
    "C (100 candidatas)": pd.read_csv(VARIANTS_DIR / "experiment-matrix_n100.csv"),
}
best_params = {
    "A": json.load(open(RESULTS / "best_params.json")),
    "B": json.load(open(VARIANTS_DIR / "best_params_seed2.json")),
    "C": json.load(open(VARIANTS_DIR / "best_params_n100.json")),
}
candidates = {
    "A": pd.read_csv(RESULTS / "tuning-candidates.csv"),
    "B": pd.read_csv(VARIANTS_DIR / "tuning-candidates_seed2.csv"),
    "C": pd.read_csv(VARIANTS_DIR / "tuning-candidates_n100.csv"),
}


def f1_by_experiment(results):
    """F1 macro of each experiment of the tree models, indexed by KEY."""
    return results[results["model"].isin(TREE_MODELS)].set_index(KEY)["f1_macro"]


f1_default = f1_by_experiment(default)
f1_tuned = {name: f1_by_experiment(r).reindex(f1_default.index) for name, r in matrices.items()}
print(f"{len(f1_default)} experimentos de árboles en cada matriz")

## Las variantes
Las tres variantes solo se diferencian en la seed y en el número de candidatas.

In [ ]:
pd.DataFrame(
    {
        "seed": [SEED, VARIANTS["seed2"]["seed"], SEED],
        "candidatas": [TUNING_N_CANDIDATES, VARIANTS["seed2"]["n_candidates"], VARIANTS["n100"]["n_candidates"]],
    },
    index=["A (30 candidatas)", "B (otra seed)", "C (100 candidatas)"],
)

## Resultados en el test
Se han calculado la métrica F1 macro en el test de cada árbol, con los valores por defecto y con cada ajuste (media de todos los experimentos).

La segunda tabla es la diferencia con los valores por defecto (sin aplicar hiperparámetros). Es positiva si el ajuste mejora.

In [ ]:
mean_f1 = pd.DataFrame({"por defecto": f1_default.groupby("model").mean()} | {n: s.groupby("model").mean() for n, s in f1_tuned.items()})
mean_f1.round(4)

In [ ]:
difference = mean_f1.drop(columns="por defecto").sub(mean_f1["por defecto"], axis=0)
difference.round(4)

Mostramos los resultados en un gráfico.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
width = 0.26
x = np.arange(len(difference))

for i, (name, color) in enumerate(zip(difference.columns, SERIES)):
    ax.bar(x + (i - 1) * width, difference[name], width, color=color, label=name)
ax.axhline(0, color=SERIE_BASE, linewidth=1, linestyle="--")
ax.set_xticks(x, difference.index)
ax.set_ylabel("F1 macro ajustado menos por defecto (test)")
ax.legend()
plt.tight_layout()
save_figure("diferencia_por_variante", "0.13-robustez-ajuste", fig=fig)
plt.show()

Ninguna de las tres ejecuciones supera a los valores por defecto. Con 100 candidatas (C) no mejora, así que la búsqueda no parece quedarse corta. Y con otra seed (B) el resultado ya cambia respecto a A, lo que apunta al ruido (C).

## Cambio del resultado según la seed
Vamos a comparar, experimento a experimento, el ajuste A con los valores por defecto y con el ajuste B, que solo cambia la seed.

In [ ]:
a = f1_tuned["A (30 candidatas)"]
gap = pd.DataFrame(
    {
        "A frente a por defecto": (a - f1_default).abs().groupby("model").mean(),
        "A frente a B (solo cambia la seed)": (a - f1_tuned["B (otra seed)"]).abs().groupby("model").mean(),
    }
)
gap.round(4)

Entre A y B, que solo cambian la seed, la diferencia es de unos 0,008 a 0,010 de F1 macro, parecida a la que hay entre A y los valores por defecto (de 0,007 a 0,016).
Cambiar la seed mueve los resultados tanto como el propio ajuste.

## Variabilidad de los valores elegidos
Comprobamos cuántos de los 153 ajustes de los árboles tienen los mismos hiperparámetros en dos variantes.

In [ ]:
tree_keys = list(best_params["B"])
same = pd.Series(
    {
        "A y B (otra seed)": sum(best_params["A"][k] == best_params["B"][k] for k in tree_keys),
        "A y C (100 candidatas)": sum(best_params["A"][k] == best_params["C"][k] for k in tree_keys),
    },
    name=f"coinciden de {len(tree_keys)}",
)
same.to_frame()

Es decir, que al cambiar de seed (B), no coincide ningún valor de los hiperparámetros.

Mientras que al subir a 100 candidatas (C) solo coinciden 15.

## Comparación de la F1 interna con la del test
Al ajustar, puntuamos cada candidata con una validación dentro de los datos de entrenamiento (la F1 macro interna) y nos quedamos con la que mejor sale. Después medimos la elegida en el test, que no ha visto nunca (la F1 macro del test).

Si la validación interna fuera fiable, una candidata con mejor F1 interna debería dar también mejor F1 en el test. Si no pasa, es que la validación es ruidosa y estamos eligiendo por suerte.

La tabla muestra las dos F1 de la candidata elegida, como media de cada modelo.

In [ ]:
def internal_score(table):
    """Mean inner-validation F1 macro of the chosen candidate, per tree model."""
    chosen = table[table["selected"] & table["model"].isin(TREE_MODELS)]
    return chosen.groupby("model")["f1_macro_mean"].mean()


inner = pd.DataFrame({name: internal_score(candidates[name[0]]) for name in matrices})
test = pd.DataFrame({name: s.groupby("model").mean() for name, s in f1_tuned.items()})
pd.concat({"F1 macro interna (validación)": inner, "F1 macro en el test": test}, axis=1).round(4)

Entre A y B la F1 interna casi no cambia (en LightGBM es 0,897 en las dos), pero la del test sí, así que la interna no nos sirve para saber qué pasará en el test.

Con 100 candidatas (C) la F1 interna sube (0,904) y la del test empeora, es decir, que buscando más solo hemos encontrado candidatas que puntúan mejor en la validación interna.

Por lo que parece que la búsqueda se ajusta al ruido de la validación interna.

## Conclusión sobre Optuna
Optuna haría una búsqueda guiada, útil para la primera explicación pero no para la segunda, porque solo encontraría mejor la candidata con mejor F1 macro interna, y esa F1 macro es ruidosa.

Por lo que decidimos no aplicar Optuna. Esto no demuestra que no ayude, porque solo hemos probado otra seed y más candidatas aleatorias, y todo se ha medido con la F1 macro.

## Resumen
- En los árboles el ajuste no mejora a los valores por defecto.
- Con más candidatas (C) sube la F1 macro interna, pero no la del test, así que la búsqueda no parece el problema.
- Con otra seed (B) cambian los resultados y los valores elegidos, tanto como el propio ajuste, lo que apunta al ruido de la validación interna.
- De momento no hemos añadido Optuna.
- La comparación en el test entre los valores por defecto y los ajustados está en el [notebook 0.12](../0.12-tuned-experiment-matrix/).